In [806]:
class Chunk:
    def __init__(self, words_list, score, words_score, check_size):
        self.chunk_size = len(words_list)
        self.words_list = words_list
        self.score = score
        self.words_score = words_score

        self.last_words_list = words_list[-check_size+1:]
        self.first_words_list = words_list[:check_size-1]
        self.sentence = " ".join(words_list)

In [807]:
x = Chunk(["asd", "asden", "talaatOssod"], 0.3, [1,2,3], 3)

In [808]:
x.score

0.3

In [809]:
WORDS_SET_PKL_PATH = "/home/mohamed/Mohamed/Vodafone_project/projects/app/files/spell_corrector_files/address_wrods_set.pkl"
PROBS_DICTS_PKL_PATH = "/home/mohamed/Mohamed/Vodafone_project/projects/app/files/spell_corrector_files/forward_bacward_probs.pkl"

In [810]:
import pickle
import fuzzywuzzy
from fuzzywuzzy import process
import itertools
import operator

In [811]:
with open(WORDS_SET_PKL_PATH, "rb") as file:
    words_set = pickle.load(file)

with open(PROBS_DICTS_PKL_PATH, "rb") as file:
    forward_probabilities, backward_probabilities = pickle.load(file)

In [812]:
GOVS_SET = {
 'الجيزه',
 'قنا',
 'القاهرة',
 'اسيوط',
 'الاسكندرية',
 'دمياط',
 'كفر الشيخ',
 'الدقهلية',
 'بورسعيد',
 'الاسماعلية',
 'شمال',
 'سيناء',
 'الوادي', 
 'الجديد',
 'الاقصر',
 'الفيوم',
 'سوهاج',
 'البحيرة',
 'جنوب',
 'سيناء',
 'السويس',
 'الشرقية',
 'المنوفية',
 'القليوبية',
 'بنى',
 'سويف',
 'المنيا',
 'اسوان',
 'البحر',
 'الاحمر',
 'الغربية',
 'جيزه',
 'قاهرة',
 'اسكندرية',
 'دمياط',
 'كفر',
 'الشيخ',
 'دقهلية',
 'بورسعيد',
 'اسماعلية',
 'شمال',
 'سيناء',
 'الوادي'
 'الجديد',
 'اقصر',
 'فيوم',
 'بحيرة',
 'سويس',
 'شرقية',
 'منوفية',
 'قليوبية',
 'بنى سويف',
 'منيا',
 'اسوان',
 'بحر',
 'احمر',
 'مطروح',
 'غربية',
}

In [829]:
def get_potential_words_prob(sentence: str, words_set: set[str], limit: int = 10):
    def inverse_dist(a, b):
        x = fuzzywuzzy.StringMatcher.distance(a, b, weights=(3, 9, 6))
        if x == 0:
            return 1.0
        return 1 / (x)
    
    input_words_list = sentence.split(" ")
    best_matchs_list = []
    for word in input_words_list:
        best_match1 = process.extractBests(
            word, words_set, scorer=inverse_dist, limit=limit
        )
        best_match2 = process.extractBests(
            word, GOVS_SET, scorer=inverse_dist, limit=limit//3
        )
        # print(best_match1+best_match2)
        best_matchs_list.append(best_match1+best_match2)

    return best_matchs_list

In [830]:
['شارع مستشفى الإيمان الجليدة أسيوط',
 'آخر مكرم عبيد مدينة نصر القاهرة',
 'شارع صلاح سالم مصر القديمة القهر',
 'شارع بورسعيد المنزلة الد أهلية',
 'آخر مكرم عبيت مدينة نصر القاهرة',
 'شارع مستشفى الإيمان الجديدة أسيوط',
 'شارع العباسي المنصورة الدهلي',
 'شارع أمين سامي القصر العيني القاهرة',
 'شارع مكتب العمل مدغمر الداهلي',
 'شارع محمد فريد مع حسن رضوان طنطا الغربية',
 'شارع عمر بن الخطاب المنصورة دهلي',
 'شارع محمد فريد مع حسن رضوان طنطا الغروية']

['شارع مستشفى الإيمان الجليدة أسيوط',
 'آخر مكرم عبيد مدينة نصر القاهرة',
 'شارع صلاح سالم مصر القديمة القهر',
 'شارع بورسعيد المنزلة الد أهلية',
 'آخر مكرم عبيت مدينة نصر القاهرة',
 'شارع مستشفى الإيمان الجديدة أسيوط',
 'شارع العباسي المنصورة الدهلي',
 'شارع أمين سامي القصر العيني القاهرة',
 'شارع مكتب العمل مدغمر الداهلي',
 'شارع محمد فريد مع حسن رضوان طنطا الغربية',
 'شارع عمر بن الخطاب المنصورة دهلي',
 'شارع محمد فريد مع حسن رضوان طنطا الغروية']

In [831]:
sent =  "شارع محمد فريد مع حسن رضوان طنطا الغربية"

In [832]:
poten_words = get_potential_words_prob(sent, words_set, limit=20)


In [833]:
def get_chunk_objs_list(
            chunk,
            forward_probabilities,
            backward_probabilities,
            check_size=None,
            spell_probs_weight=1.0,
        ):

    chunk_size = len(chunk)
    if check_size == None:
        check_size = chunk_size

    posibilities = list(itertools.product(*chunk))
    chunk_objs_list = []

    for _, posibility in enumerate(posibilities):

        occur_probs = [0.0 for _ in range(chunk_size)]
        spell_probs = [spell_probs_weight for _ in range(chunk_size)]

        words = []
        words_score = []

        for j, (word, spell_prob) in enumerate(posibility):

            words.append(word)
            words_score.append(spell_prob)

            spell_probs[j] *= spell_prob

            for k in range(j + 1, len(posibility)):

                occur_probs[j] += forward_probabilities[
                    (word, posibility[k][0])
                ]
                occur_probs[k] += backward_probabilities[
                    (posibility[k][0], word)
                ]

        probs = list(map(operator.mul, occur_probs, spell_probs))

        all_probs = 0
        for prob in probs:
            all_probs += prob
        all_probs /= chunk_size

        chunk_obj = Chunk(words, all_probs, words_score, check_size=check_size)

        chunk_objs_list.append(chunk_obj)

    return chunk_objs_list

In [834]:
x = get_chunk_objs_list(
            poten_words[:3],
            forward_probabilities,
            backward_probabilities,
)

In [835]:
def get_sentece_chunks(sentence, chunk_size=3):
    potential_words = get_potential_words_prob(sentence, words_set=words_set, limit=8)
    list_of_chunk_objs_list = []
    for i in range(len(potential_words)- chunk_size+1):
        chunk_objs_list = get_chunk_objs_list(
            potential_words[i:chunk_size+i],
            forward_probabilities,
            backward_probabilities,
            spell_probs_weight=0.5,
        )
        list_of_chunk_objs_list.append(chunk_objs_list)
    return list_of_chunk_objs_list

    


In [836]:
import copy

In [837]:
def merge_chunks_bacward(chunk, beam_leafs, check_size):
    tranformed_chunk = []
    curr = []
    for leaf in beam_leafs:
        curr.append( (leaf.words_list[0], leaf.words_score[0]))

    tranformed_chunk.append(curr)

    for i in range(0, chunk.chunk_size):
        tranformed_chunk.append([(chunk.words_list[i], chunk.words_score[i])])

    return get_chunk_objs_list(
            tranformed_chunk,
            forward_probabilities,
            backward_probabilities,
            check_size=check_size,
            spell_probs_weight=0.5,
        )

def merge_chunks_forward(chunk, beam_leafs, check_size):
    tranformed_chunk = []
    

    for i in range(0, chunk.chunk_size):
        tranformed_chunk.append([(chunk.words_list[i], chunk.words_score[i])])

    curr = []
    for leaf in beam_leafs:
        curr.append( (leaf.words_list[-1], leaf.words_score[-1]))

    tranformed_chunk.append(curr)

    x = get_chunk_objs_list(
            tranformed_chunk,
            forward_probabilities,
            backward_probabilities,
            check_size=check_size,
            spell_probs_weight=0.5,
        )
    
    # for y in x:
    #     print(y.score)
    #     print(y.words_list)


    return x
    


In [838]:
def beam_search(list_of_chunk_objs_list, top_k=5, chunk_size=3):
    size = len(list_of_chunk_objs_list)
    all_bests = []
    for i in range(size):
        # Get best k 
        best_k = sorted(list_of_chunk_objs_list[i], key=lambda x: x.score, reverse=True)[:top_k]
        curr_best_k = copy.deepcopy(best_k)
        direction = True
        fi = i+1
        bi = i-1
        while fi < size and bi >= 0:
            new_best_k = []
            if direction:
                for j, chunk in enumerate(curr_best_k):
                    # Filter from back
                    beam_leafs = list(
                        filter(lambda x: x.last_words_list == chunk.first_words_list, list_of_chunk_objs_list[bi])
                    )
                    beam_leafs = sorted(beam_leafs, key=lambda x: x.score, reverse=True)[:top_k]
                    # Merge chunk with beam leafs 
                    new_best_k += merge_chunks_bacward(chunk, beam_leafs, check_size=chunk_size)

                bi-=1

            else: 
                for j, chunk in enumerate(curr_best_k):
                    # Filter from back
                    beam_leafs = list(
                        filter(lambda x: x.first_words_list == chunk.last_words_list, list_of_chunk_objs_list[fi])
                    )
                    beam_leafs = sorted(beam_leafs, key=lambda x: x.score, reverse=True)[:top_k]
                    # Merge chunk with beam leafs 
                    new_best_k += merge_chunks_forward(chunk, beam_leafs, check_size=chunk_size)
                fi+=1

            direction = not direction
            curr_best_k = sorted(new_best_k, key=lambda x: x.score, reverse=True)[:top_k]

        while fi < size:
            new_best_k = []
            for j, chunk in enumerate(curr_best_k):
                # Filter from back
                beam_leafs = list(
                    filter(lambda x: x.first_words_list == chunk.last_words_list, list_of_chunk_objs_list[fi])
                )
                beam_leafs = sorted(beam_leafs, key=lambda x: x.score, reverse=True)[:top_k]
                # Merge chunk with beam leafs 
                new_best_k += merge_chunks_forward(chunk, beam_leafs, check_size=chunk_size)
            fi+=1

            curr_best_k = sorted(new_best_k, key=lambda x: x.score, reverse=True)[:top_k]




        while bi >= 0:
            new_best_k = []
            for j, chunk in enumerate(curr_best_k):
                # Filter from back
                beam_leafs = list(
                    filter(lambda x: x.last_words_list == chunk.first_words_list, list_of_chunk_objs_list[bi])
                )
                beam_leafs = sorted(beam_leafs, key=lambda x: x.score, reverse=True)[:top_k]
                # Merge chunk with beam leafs 
                new_best_k += merge_chunks_bacward(chunk, beam_leafs, check_size=chunk_size)
            bi-=1

            curr_best_k = sorted(new_best_k, key=lambda x: x.score, reverse=True)[:top_k]
        
        all_bests += curr_best_k

    return sorted(all_bests, key=lambda x: x.score, reverse=True)[:top_k]


        



            

In [839]:
res = get_sentece_chunks('شارع مستشفى الإيمان الجديدة أسيوط', chunk_size=3)
z = beam_search(res, top_k=5, chunk_size=3)


In [840]:
for a in z:
    print(a.score)
    print(a.words_list)
    print(a.words_score)


0.307455305347475
['شارع', 'مستشفى', 'الإيمان', 'الجديدة', 'اسيوط']
[1.0, 1.0, 1.0, 1.0, 0.16666666666666666]
0.307455305347475
['شارع', 'مستشفى', 'الإيمان', 'الجديدة', 'اسيوط']
[1.0, 1.0, 1.0, 1.0, 0.16666666666666666]
0.2981033990717136
['شارع', 'مستشفى', 'الإيمان', 'الجديدة', 'دمياط']
[1.0, 1.0, 1.0, 1.0, 0.05555555555555555]
0.2865253267173313
['شارع', 'مستشفى', 'الإيمان', 'الجديدة', 'السيوطى']
[1.0, 1.0, 1.0, 1.0, 0.08333333333333333]
0.2809685740802605
['شارع', 'مستشفى', 'الإيمان', 'الجديدة', 'أسيوط']
[1.0, 1.0, 1.0, 1.0, 1.0]


In [841]:
sens = [['شارع مستشفى الإيمان الجديدة أسيوط'],
 ['آخر مكرم عبيد مدينة نصر القاهرة'],
 ['شارع صلاح سالم مصر القديمة القاهر'],
 ['شارع بورسعيد المنزلة الدقهلية'],
 ['آخر مكرم عبيد مدينة نصر القاهرة'],
 ['شارع مستشفى الإيمان الجديدة أسيوط'],
 ['شارع العباسي المنصورة الاهلي'],
 ['شارع أمين سامي القصر العيني القاهرة'],
 ['شارع مكتب العمل ميت غمر الاهلي'],
 ['شارع محمد فريد مع حسن رضوان طنطا الغربية'],
 ['شارع عمر بن الخطاب المنصورة دهب'],
 ['شارع محمد فريد مع حسن رضوان طنطا الغربية']]

In [842]:
all = []
for sent in sens:
    res = get_sentece_chunks(sent[0], chunk_size=3)
    all.append(beam_search(res, top_k=3, chunk_size=3))


In [843]:
for al in all:
    for a in al:
        print(a.score)
        print(a.words_list)
        print(a.sentence)

        print(a.words_score)

    print()

0.307455305347475
['شارع', 'مستشفى', 'الإيمان', 'الجديدة', 'اسيوط']
شارع مستشفى الإيمان الجديدة اسيوط
[1.0, 1.0, 1.0, 1.0, 0.16666666666666666]
0.307455305347475
['شارع', 'مستشفى', 'الإيمان', 'الجديدة', 'اسيوط']
شارع مستشفى الإيمان الجديدة اسيوط
[1.0, 1.0, 1.0, 1.0, 0.16666666666666666]
0.2981033990717136
['شارع', 'مستشفى', 'الإيمان', 'الجديدة', 'دمياط']
شارع مستشفى الإيمان الجديدة دمياط
[1.0, 1.0, 1.0, 1.0, 0.05555555555555555]

0.14645991826783628
['آخر', 'مكرم', 'عبيد', 'مدينه', 'نصر', 'القاهرة']
آخر مكرم عبيد مدينه نصر القاهرة
[1.0, 1.0, 1.0, 0.16666666666666666, 1.0, 1.0]
0.14645991826783628
['آخر', 'مكرم', 'عبيد', 'مدينه', 'نصر', 'القاهرة']
آخر مكرم عبيد مدينه نصر القاهرة
[1.0, 1.0, 1.0, 0.16666666666666666, 1.0, 1.0]
0.14645991826783628
['آخر', 'مكرم', 'عبيد', 'مدينه', 'نصر', 'القاهرة']
آخر مكرم عبيد مدينه نصر القاهرة
[1.0, 1.0, 1.0, 0.16666666666666666, 1.0, 1.0]

0.38751492988791375
['شارع', 'صلاح', 'سالم', 'مصر', 'القديمه', 'القاهرة']
شارع صلاح سالم مصر القديمه القاهرة
[1.0, 